# Reasoning Loops Demo

**Validating Research Findings:**
- "Language models can overthink" (The Decoder, Jan 2025)
- "How many reasoning steps do AI agents need" (Particula, Jul 2025)
- "How to Prevent Infinite Loops" (CodiesHub, Dec 2025)

---

## 🎯 What This Demo Validates

1. **Agents call same tool repeatedly** without making progress
2. **Debounce hooks detect and block** duplicate calls
3. **Clear success states** help agents know when to stop
4. **Hard limits** prevent runaway execution

---

## 📦 Setup

In [ ]:
import os
import time
os.environ['OTEL_SDK_DISABLED'] = 'true'

from dotenv import load_dotenv
from strands import Agent
from strands.models.openai import OpenAIModel
from tools import search_flights, check_hotel_price, book_flight, book_hotel
from hooks import DebounceHook, LimitToolCounts

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("⚠️ OPENAI_API_KEY not set. Get yours at https://platform.openai.com/api-keys and add OPENAI_API_KEY=your-key to a .env file.")

MODEL = OpenAIModel(model_id="gpt-4o-mini")

# Prompt that causes the agent to retry when it can't find prices within budget —
# this is what triggers organic loops in Scenario 1.
PERSISTENT_PROMPT = (
    "You are a persistent travel agent. Always try to find prices within the user's budget. "
    "If results are over budget, search again — prices fluctuate and you might find better deals on retry."
)

# Scenarios 1 and 2 use the exact same query. Only difference: hooks.
BUDGET_QUERY = "Find me the cheapest flight from NYC to Paris under $400 and a hotel under $200/night for March 15"

def count_tool_calls(agent):
    count = 0
    for msg in agent.messages:
        for block in msg.get("content", []):
            if "toolUse" in block:
                count += 1
    return count

print("✅ Setup complete!")

---

## 🔬 Scenario 1: Baseline (No Loop Detection)

**Research:** "Agent calls same tool repeatedly without progress"

**Expected:** May make redundant calls without detection

In [ ]:
agent_loop = Agent(
    model=MODEL,
    system_prompt=PERSISTENT_PROMPT,
    tools=[search_flights, check_hotel_price],
)

start = time.time()
response = agent_loop(BUDGET_QUERY)
time_loop = time.time() - start
calls_loop = count_tool_calls(agent_loop)

print(f"⏱️  {time_loop:.1f}s — {calls_loop} tool calls")
if calls_loop > 4:
    print(f"⚠️  {calls_loop} calls — ambiguous feedback caused retries")
else:
    print("ℹ️  Agent stopped early (LLM behavior varies run-to-run)")

---

## 🚫 Scenario 2: Debounce Hook (Solution from Research)

**Research Solution:** "Cache/Debounce Layer with Hooks"

**Expected:** Duplicate calls detected and blocked

In [ ]:
debounce = DebounceHook(window_size=3)

agent_debounce = Agent(
    model=MODEL,
    system_prompt=PERSISTENT_PROMPT,
    tools=[search_flights, check_hotel_price],
    hooks=[debounce],  # only change from Scenario 1
)

start = time.time()
response = agent_debounce(BUDGET_QUERY)
time_debounce = time.time() - start

stats = debounce.get_stats()
calls_debounce = stats['total_calls']

print(f"⏱️  {time_debounce:.1f}s — {stats['total_calls']} allowed, {stats['blocked_calls']} blocked")
if stats['blocked_calls'] > 0:
    print(f"✅ DebounceHook blocked {stats['blocked_calls']} duplicate calls")
else:
    print("✅ No duplicates this run (LLM behavior varies)")

---

## ✅ Scenario 3: Clear Success States

**Research:** "Tools return SUCCESS/FAILED, agent knows when to stop"

**Expected:** Agent stops after receiving SUCCESS

In [ ]:
agent_clear = Agent(
    model=MODEL,
    tools=[book_flight, book_hotel],
)

query_book = "Book a flight NYC to Paris for Alex Rivera, and a hotel called Le Marais for 3 nights"

start = time.time()
response = agent_clear(query_book)
time_clear = time.time() - start
calls_clear = count_tool_calls(agent_clear)

print(f"⏱️  {time_clear:.1f}s — {calls_clear} tool calls")
print("✅ SUCCESS states — agent stopped immediately")

---

## 🔢 Scenario 4: Hard Limits

**Research:** "Treat every agent run as bounded process with explicit limits"

**Expected:** Agent stops at reasonable iteration count

In [ ]:
limit_hook = LimitToolCounts(max_tool_counts={
    "search_flights": 2,
    "check_hotel_price": 2,
})

agent_limits = Agent(
    model=MODEL,
    system_prompt="You are a travel agent. Find the best deal for the user.",
    tools=[search_flights, check_hotel_price],
    hooks=[limit_hook],
)

query_multi = "Compare flights and hotels for NYC to Paris, London, and Tokyo — find the cheapest option for each"

start = time.time()
response = agent_limits(query_multi)
time_limits = time.time() - start
calls_limits = sum(limit_hook.tool_counts.values())

print(f"⏱️  {time_limits:.1f}s — tool counts: {limit_hook.tool_counts}")
print("✅ Hard ceiling enforced")

In [ ]:
debounce_stats = debounce.get_stats()

print(f"{'Scenario':<35} {'Calls':>6} {'Blocked':>8} {'Time':>8}")
print("-"*60)
print(f"{'1. Ambiguous feedback':<35} {calls_loop:>6} {'—':>8} {time_loop:>6.1f}s")
print(f"{'2. DebounceHook':<35} {calls_debounce:>6} {debounce_stats['blocked_calls']:>8} {time_debounce:>6.1f}s")
print(f"{'3. Clear SUCCESS states':<35} {calls_clear:>6} {'—':>8} {time_clear:>6.1f}s")
print(f"{'4. LimitToolCounts':<35} {calls_limits:>6} {'2+':>8} {time_limits:>6.1f}s")

if calls_loop > calls_clear:
    print(f"\n→ Ambiguous: {calls_loop} calls  vs  Clear states: {calls_clear} calls")
if debounce_stats['blocked_calls'] > 0:
    print(f"→ DebounceHook blocked {debounce_stats['blocked_calls']} duplicates")
print(f"→ LimitToolCounts: ceiling of 2 per tool enforced")